In [70]:
import nltk
import random
import string

from nltk.corpus import wordnet
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

def remove_stop_words(input_text):
    """
    """
    stop_words = set(stopwords.words("english"))
    tokens = word_tokenize(input_text)
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text


def remove_punctuations(input_text):
    filtered_text = input_text.translate(str.maketrans('', '', string.punctuation))
    return filtered_text

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [65]:
nltk.download('averaged_perceptron_tagger')
from translate import Translator

def back_translate(text, target_language='fr', source_language='en'):
    translator = Translator(to_lang=target_language, from_lang=source_language)
    
    # Translate the text to the target language
    translated_text = translator.translate(text)
    
    # Translate the translated text back to the source language
    back_translated_text = translator.translate(translated_text, to_lang=source_language, from_lang=target_language)
    
    return back_translated_text
def synonym_replacement(sentence, n=5):
    # Remove the punctuations 
    filtered_sentence = remove_punctuations(sentence) 

    # Remove the stop words 
    filtered_sentence = remove_stop_words(filtered_sentence)

    words = nltk.word_tokenize(filtered_sentence)
    for _ in range(n):
        word_to_replace = random.choice(words)
        synonyms = wordnet.synsets(word_to_replace)
        if synonyms:
            synonym = random.choice(synonyms).lemma_names()[0]
            sentence = sentence.replace(word_to_replace, synonym)
            
    return sentence

def random_insertion(sentence, n=1): 
    # Remove the punctuations 
    filtered_sentence = remove_punctuations(sentence) 

    # Remove the stop words 
    filtered_sentence = remove_stop_words(filtered_sentence)
    words = nltk.word_tokenize(filtered_sentence)
    for _ in range(n):
        word_to_insert = random.choice(words)
        word_to_insert_with = random.choice(words)
        sentence = sentence.replace(word_to_insert_with, word_to_insert_with + ' ' + word_to_insert)
    return sentence

def random_deletion(sentence, n=2): 
    # Remove the punctuations 
    filtered_sentence = remove_punctuations(sentence) 

    # Remove the stop words 
    filtered_sentence = remove_stop_words(filtered_sentence)
    words = nltk.word_tokenize(filtered_sentence)
    for _ in range(n):
        word_to_delete = random.choice(words)
        sentence = sentence.replace(word_to_delete,'')

    return sentence


def random_masking(sentence, n=1):
    # Remove the punctuations 
    filtered_sentence = remove_punctuations(sentence) 

    # Remove the stop words 
    filtered_sentence = remove_stop_words(filtered_sentence)
    words = nltk.word_tokenize(filtered_sentence)    
    for _ in range(n):
        word_to_mask = random.choice(words)
        sentence = sentence.replace(word_to_mask, "<MASK>")

    return sentence

def textual_entailment(sentence, n = 5):
    # Example: Change active voice to passive voice

    tokens = nltk.word_tokenize(sentence)
    tagged = nltk.pos_tag(tokens)
    random.shuffle(tagged)
    tagged = tagged[0:n]

    for word, tag in tagged:
        if tag.startswith('VB'):  # Verbs
            morhped_word = wordnet.morphy(word, wordnet.VERB)
            if morhped_word != None:
                new_word = morhped_word
            else:
                new_word = word
            sentence = sentence.replace(word, new_word)
        elif tag.startswith('NN'):  # Nouns
            morhped_word = wordnet.morphy(word, wordnet.NOUN)
            if morhped_word != None:
                new_word = morhped_word
            else:
                new_word = word
            sentence = sentence.replace(word, new_word)
        else:
            continue
        
    return sentence


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [ ]:
from datamodules.EmbeddingData import EmbeddingData


dm = EmbeddingData(batch_size=4,
                   data_split="merged_1",
                   LLM_name="all-mpnet-base-v1",
                   augmentations=['synonym replacement','random insertion','random swap'],
                   num_aug_words=0,
                   num_classes=51,
                   )

print(dm.augmentations)


In [ ]:
train_df = dm.train_data_df
print(train_df.size)
print(train_df.iloc[1900])


In [ ]:
import pandas as pd

df_for_current_aug = train_df
df_for_current_aug['abstractText'].apply(synonym_replacement)

#df_for_current_aug['abstractText'] = train_df['abstractText'].apply(synonym_replacement)
train_df = pd.concat([train_df, df_for_current_aug], axis=0)

del df_for_current_aug
print(train_df.size)



In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

input_file_path = "/workspace/code/data/astmh2023AbstractContents_26mar2024.xlsx"

In [4]:
# Read Excel file
df = pd.read_excel(input_file_path)

# Define features and labels
abs_title = df[['title']]
abs_text = df[['abstractText']]
merged_categories = df[['mergedCategory']]
short_gen_categories = df['shortGenCat']

print(df['mergedCategory'])

0                           Arthropods/Entomology - Other
1                           Arthropods/Entomology - Other
2                           Arthropods/Entomology - Other
3                           Arthropods/Entomology - Other
4                           Arthropods/Entomology - Other
                              ...                        
2273    Water, Sanitation, Hygiene and Environmental H...
2274    Water, Sanitation, Hygiene and Environmental H...
2275    Water, Sanitation, Hygiene and Environmental H...
2276    Water, Sanitation, Hygiene and Environmental H...
2277    Water, Sanitation, Hygiene and Environmental H...
Name: mergedCategory, Length: 2278, dtype: object


In [6]:
import re
sentence_end_pattern = re.compile(r'(?<=[.!?])\s+')

def print_text_linebyline(input_text):
    sentences = sentence_end_pattern.split(input_text)
    for text_line in sentences:
        print(text_line)

In [53]:
import random

# Randomly select an entry in the spread sheet. 

random_idx = random.randint(0, len(abs_title))
rand_abs_title = abs_title.iloc[random_idx]['title']
rand_abs_text = abs_text.iloc[random_idx]['abstractText']
rand_abs_category = merged_categories.iloc[random_idx]['mergedCategory']

print_text_linebyline(rand_abs_text)


Leishmaniasis is a vector disease caused by protozoa of the genus Leishmania that affect humans producing a wide spectrum of clinical forms.
Recently, the association of some Leishmania species with the presence of an endosymbiont double-stranded ribonucleic acid virus from the Totiviridae family called leishmaniavirus (LRV) have been described, showing increased pathology and treatment failures in individuals with cutaneous and mucocutaneous forms of the disease.
Two types of LRV have been describes, type 1 circulates in the Americas while type 2 in the Old World.
In America, the presence of LRV has been reported in Leishmania isolates from French Guiana, Brazil, Peru, Colombia, Ecuador, and Costa Rica.
In a recent pilot study in Panama, 10 L_(V.) panamensis LRV-1+ isolates were detected by RT-PCR.
In Panama, it is unknown if there are other species of Leishmania Viannia (L.
(V.) guyanensis and L_(V.) braziliensis) associated with the presence of LRV-1.
Therefore, the aim of this stud

In [23]:
import difflib

def highlight_text_differences(text1, text2):
    # Perform a diff between the two texts
    differ = difflib.Differ()
    diff = list(differ.compare(text1.splitlines(), text2.splitlines()))

    # Highlight the differences
    highlighted_diff = []
    for line in diff:
        if line.startswith('-'):
            highlighted_diff.append(f"\033[91m{line}\033[0m")  # Red color for deletions
        elif line.startswith('+'):
            highlighted_diff.append(f"\033[92m{line}\033[0m")  # Green color for additions
        else:
            highlighted_diff.append(line)

    return '\n'.join(highlighted_diff)

In [ ]:
# Text pre-processing
print("Text :", rand_abs_text)
print("Text without punctuation:", remove_punctuations(rand_abs_text)) # synonym_replacement, random_insertion, random_deletion, random_swap, random_masking, textual_entailment
print("Text without stopwords:", remove_stop_words(rand_abs_text))

In [69]:
ori_text = rand_abs_text
augmented_text = back_translate( "This is a sample text for back translation.", target_language='fr', source_language='en')
#print_text_linebyline(ori_text)
#print("####     ####")
#print_text_linebyline(augmented_text)

print(highlight_text_differences(ori_text, augmented_text))

TypeError: translate() got an unexpected keyword argument 'to_lang'

In [ ]:
augmented_text = random_insertion(ori_text, n=2)
print('Random Insertion:', augmented_text)

In [ ]:
augmented_text = random_deletion(ori_text, p=0.3)
print('Random Deletion:', augmented_text)

In [ ]:
augmented_text = random_swap(ori_text, n=2)
print('Random Swap:', augmented_text)